# 01. 객실 기록의 의미와 승객 등급의 관계

객실 위치가 생존과 관련 있다면 승객 등급 외에 어떤 정보를 더 줄 수 있을까?
모델에 넣기 전에 Cabin의 결측과 복합 표기, 등급별 분포를 확인했다.

여러 객실 번호가 인접한 객실의 유사한 위치 정보를 담을 가능성을 생각했지만,
번호만으로 실제 인접성이나 같은 위치를 확정할 수는 없다. 먼저 실제 기록을 확인했다.


> **기록 상태**: 2026년 7월 개인 실험을 2026년 10월 공개용으로 정리한 노트북입니다.
> 2026-10-06 정리본을 Kaggle 원자료(`train.csv`, `test.csv`)로 Google Colab에서 처음부터 다시 실행했고,
> 정리 전 원본에 저장돼 있던 표와 수치가 모두 같은 값으로 재현됨을 확인했습니다. 실행 환경은 마지막 셀에 기록했습니다.
> 원본·수정 범위는 [기록 정리 내역](../reports/provenance.md)에 있습니다.


## 1. 데이터와 결측 확인


In [5]:
import numpy as np
import pandas as pd


In [ ]:
from pathlib import Path
import os

# 로컬: 저장소 루트 또는 notebooks/에서 실행. Kaggle: Titanic 데이터를 연결.
# Colab: train.csv와 test.csv를 /content에 올린 뒤 실행.
# 다른 위치를 쓰려면 TITANIC_DATA_DIR에 train.csv/test.csv가 있는 폴더를 지정한다.
configured_dir = os.environ.get("TITANIC_DATA_DIR")
if configured_dir:
    candidates = [Path(configured_dir)]
else:
    candidates = [
        Path("data"), Path("../data"),
        Path("/kaggle/input/competitions/titanic"),
        Path("/kaggle/input/titanic"),
        Path("/content"),
    ]
data_dir = next(
    (p for p in candidates if (p / "train.csv").is_file() and (p / "test.csv").is_file()),
    None,
)
if data_dir is None:
    raise FileNotFoundError(
        "Titanic train.csv와 test.csv를 data/에 넣거나 TITANIC_DATA_DIR을 지정하세요."
    )
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
assert "Survived" in train.columns and "Survived" not in test.columns
assert train["PassengerId"].is_unique and test["PassengerId"].is_unique
print("train:", train.shape, "test:", test.shape)


In [7]:
data_summary = pd.DataFrame({
    "dtype": train.dtypes,
    "missing_count": train.isna().sum(),
    "missing_rate": train.isna().mean(),
    "nunique": train.nunique()
})

data_summary.sort_values("missing_rate", ascending=False)

,dtype,missing_count,missing_rate,nunique
Cabin,object,687,0.771044,147
Age,float64,177,0.198653,88
Embarked,object,2,0.002245,3
PassengerId,int64,0,0.000000,891
Name,object,0,0.000000,891
Pclass,int64,0,0.000000,3
Survived,int64,0,0.000000,2
Sex,object,0,0.000000,2
Parch,int64,0,0.000000,7
SibSp,int64,0,0.000000,7


In [8]:
df = train.copy()
df["CabinKnown"] = df["Cabin"].notna().astype(int)


## 2. 여러 토큰이 적힌 Cabin 기록

공백이 있는 Cabin 행을 추출해 표기를 직접 확인한다. 이 조건으로 찾은 24행을 모두
"여러 객실을 예약한 승객"으로 단정하지 않는다. 복합 표기를 찾기 위한 탐색 조건이다.


In [9]:
multiple_cabins = df.loc[
    df["Cabin"].fillna("").str.contains(r"\s"),
    ["PassengerId", "Name", "Cabin"]
]

print("여러 객실이 기록된 행 수:", len(multiple_cabins))
multiple_cabins

여러 객실이 기록된 행 수: 24


,PassengerId,Name,Cabin
27,28,"Fortune, Mr. Charles Alexander",C23 C25 C27
75,76,"Moen, Mr. Sigurd Hansen",F G73
88,89,"Fortune, Miss. Mabel Helen",C23 C25 C27
97,98,"Greenfield, Mr. William Bertram",D10 D12
118,119,"Baxter, Mr. Quigg Edmond",B58 B60
128,129,"Peter, Miss. Anna",F E69
297,298,"Allison, Miss. Helen Loraine",C22 C26
299,300,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",B58 B60
305,306,"Allison, Master. Hudson Trevor",C22 C26
311,312,"Ryerson, Miss. Emily Borie",B57 B59 B63 B66


### 기록을 보고 남긴 판단

`C23 C25 C27`, `B96 B98`처럼 같은 문자에 여러 번호가 이어진 사례가 있다.
반면 `F G73`, `F E69`처럼 해석을 따로 확인해야 할 표기도 있다.
가까운 번호라는 이유만으로 실제로 인접한 객실이라고 결론 내리지는 않았다.

이 실험에서는 세부 객실 번호를 위치로 환산하지 않고, 문자열 맨 앞 문자를
`Deck`이라는 단순한 대리변수로 사용했다. 따라서 `Deck`은 선박 도면으로 검증한 위치가 아니다.
복합 표기에서 첫 문자만 쓰는 규칙의 타당성은 남은 확인 사항이다.


In [10]:
df["Deck"] = (
    df["Cabin"]
      .str.extract(r"^([A-Za-z])")[0]
      .fillna("Unknown")
)

df[["Cabin", "Deck"]].head(15)

,Cabin,Deck
0,NaN,Unknown
1,C85,C
2,NaN,Unknown
3,C123,C
4,NaN,Unknown
5,NaN,Unknown
6,E46,E
7,NaN,Unknown
8,NaN,Unknown
9,NaN,Unknown


## 3. Deck별 생존율과 표본 수


In [11]:
deck_summary = (
    df.groupby("Deck")
      .agg(
          PassengerCount=("Survived", "count"),
          SurvivalRate=("Survived", "mean")
      )
      .sort_values("SurvivalRate", ascending=False)
      .reset_index()
)

deck_summary

,Deck,PassengerCount,SurvivalRate
0,D,33,0.757576
1,E,32,0.750000
2,B,47,0.744681
3,F,13,0.615385
4,C,59,0.593220
5,G,4,0.500000
6,A,15,0.466667
7,Unknown,687,0.299854
8,T,1,0.000000


`Cabin`은 891명 중 687명(77.1%)에게 기록이 없다. Deck별 생존율 차이에는
기록 누락과 승객 구성의 차이가 함께 섞일 수 있다. 특히 G는 4명, T는 1명으로 표본이 작다.


## 4. Deck과 Pclass의 겹침


In [12]:
pd.crosstab(
    df["Deck"],
    df["Pclass"],
    normalize="index"
).round(3)

Pclass,1,2,3
Deck,,,
A,1.000,0.000,0.000
B,1.000,0.000,0.000
C,1.000,0.000,0.000
D,0.879,0.121,0.000
E,0.781,0.125,0.094
F,0.000,0.615,0.385
G,0.000,0.000,1.000
T,1.000,0.000,0.000
Unknown,0.058,0.245,0.697


A·B·C로 분류된 승객은 모두 1등급이다. 따라서 초기 메모의
"Pclass와 Deck은 큰 상관이 없는 것 같다"는 추측은 이 표를 근거로 유지하기 어렵다.
다만 등급과 Deck의 겹침이 곧 예측 정보의 완전한 중복을 뜻하지는 않는다.


## 5. 같은 승객 등급 안에서도 차이가 남는가?


In [13]:
cabin_class_summary = (
    df.groupby(["Pclass", "CabinKnown"])
      .agg(
          PassengerCount=("Survived", "count"),
          SurvivalRate=("Survived", "mean")
      )
      .reset_index()
)

cabin_class_summary

,Pclass,CabinKnown,PassengerCount,SurvivalRate
0,1,0,40,0.475000
1,1,1,176,0.664773
2,2,0,168,0.440476
3,2,1,16,0.812500
4,3,0,479,0.235908
5,3,1,12,0.500000


In [14]:
deck_class_summary = (
    df.groupby(["Pclass", "Deck"])
      .agg(
          PassengerCount=("Survived", "count"),
          SurvivalRate=("Survived", "mean")
      )
      .reset_index()
)

deck_class_summary

,Pclass,Deck,PassengerCount,SurvivalRate
0,1,A,15,0.466667
1,1,B,47,0.744681
2,1,C,59,0.593220
3,1,D,29,0.758621
4,1,E,25,0.720000
5,1,T,1,0.000000
6,1,Unknown,40,0.475000
7,2,D,4,0.750000
8,2,E,4,0.750000
9,2,F,8,0.875000


## 6. 다음 비교를 정한 이유

같은 Pclass 안에서도 Cabin 기록 유무에 따른 생존율 차이가 관찰된다.
그러나 다른 승객 특성을 통제한 인과적 효과를 추정한 것은 아니다.
이에 다음 노트북에서는 로지스틱 회귀와 검증 분할을 고정하고,
객실 변수를 추가하거나 Pclass를 대체했을 때 예측 결과가 달라지는지 비교한다.

**확인한 것**: 복합 표기 24행, 많은 결측, Deck과 Pclass의 겹침, 등급 내 생존율 차이.
**확인하지 않은 것**: 객실의 실제 인접성, 첫 문자 대표 규칙의 최적성, 위치의 인과적 효과.

다음: [02. 변수의 추가 가치 비교](02_feature_comparison.ipynb)


## 실행 환경 기록

전체 재실행 시 아래 버전을 결과와 함께 보관합니다.


In [15]:
import platform
from importlib.metadata import version

print("Python", platform.python_version())
for package in ["numpy", "pandas", "scikit-learn", "matplotlib"]:
    print(package, version(package))


Python 3.13.16
numpy 2.1.3
pandas 2.2.3
scikit-learn 1.6.1
matplotlib 3.10.0
